# Headless server

> Sessions, prompts and approvals over HTTP and SSE, for a GUI that wants to be a thin client
> rather than re-implementing the approval gate. The other half of `ramabana-acp`: an editor
> drives one in-process agent over stdio; this drives any number of agents over the network.

In [ ]:
#| default_exp serve

In [ ]:
#| export
import asyncio, json, threading
from pathlib import Path

from fastcore.script import call_parse
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
from sse_starlette.sse import EventSourceResponse

from ramabana.cli import mk_agent


In [ ]:
#| export
class Sessions:
    "Every session this server is holding, each its own `Agent` rooted at the same folders."

    def __init__(self, roots, model=None, cfg=None, **kw):
        self.roots, self.model, self.cfg, self.kw = list(roots), model, cfg, kw
        self.agents = {}

    def create(self, model=None):
        "A fresh session on `model`, or this server's own default."
        agent, _ = mk_agent(self.roots, model=model or self.model, approve='ask', cfg=self.cfg, **self.kw)
        self.agents[agent.session_id] = agent
        return agent.session_id

    def get(self, sid):
        "The session named `sid`, or None."
        return self.agents.get(sid)

    def close(self, sid):
        "Close and drop a session; True when there was one."
        agent = self.agents.pop(sid, None)
        if agent is None: return False
        agent.close()
        return True

    def close_all(self):
        for sid in list(self.agents): self.close(sid)


In [ ]:
#| export
async def _list_sessions(request):
    return JSONResponse({'sessions': list(request.app.state.sessions.agents)})

async def _create_session(request):
    body = await request.json() if await request.body() else {}
    sid = request.app.state.sessions.create(model=body.get('model'))
    return JSONResponse({'session_id': sid})

async def _close_session(request):
    ok = request.app.state.sessions.close(request.path_params['sid'])
    if not ok: return JSONResponse({'error': 'no such session'}, status_code=404)
    return JSONResponse({'closed': True})

async def _plan(request):
    agent = request.app.state.sessions.get(request.path_params['sid'])
    if agent is None: return JSONResponse({'error': 'no such session'}, status_code=404)
    return JSONResponse(agent.plan.dict())

async def _answer(request):
    agent = request.app.state.sessions.get(request.path_params['sid'])
    if agent is None or agent.approvals is None: return JSONResponse({'error': 'no such session'}, status_code=404)
    body = await request.json()
    a = agent.approvals.answer(str(body.get('id', '')), bool(body.get('ok')), str(body.get('note', '')))
    if a is None: return JSONResponse({'error': 'no ask pending under that id'}, status_code=409)
    return JSONResponse(a.dict())


In [ ]:
#| export
async def _events(agent, prompt):
    "One turn as SSE events: `chunk` as text arrives, `ask`/`answered` as approvals happen, `done` last."
    loop, q = asyncio.get_running_loop(), asyncio.Queue()
    def put(kind, data): loop.call_soon_threadsafe(q.put_nowait, (kind, data))
    unhook = (agent.approvals.listen(on_ask=lambda a: put('ask', a.dict()),
                                     on_answer=lambda a: put('answered', a.dict()))
             if agent.approvals is not None else (lambda: None))
    def pump():
        try:
            for chunk in agent.stream(prompt): put('chunk', {'text': chunk})
        except Exception as e: put('error', {'message': str(e)})
        finally: put('done', None)
    threading.Thread(target=pump, daemon=True, name='ramabana-serve-turn').start()
    try:
        while True:
            kind, data = await q.get()
            if kind == 'done':
                yield {'event': 'done', 'data': json.dumps({'reply': agent.history[-1].get('reply', '') if agent.history else ''})}
                return
            yield {'event': kind, 'data': json.dumps(data)}
    finally:
        unhook()

async def _prompt(request):
    agent = request.app.state.sessions.get(request.path_params['sid'])
    if agent is None: return JSONResponse({'error': 'no such session'}, status_code=404)
    body = await request.json()
    prompt = str(body.get('prompt', ''))
    if not prompt: return JSONResponse({'error': 'no prompt given'}, status_code=400)
    return EventSourceResponse(_events(agent, prompt))


In [ ]:
#| export
def _openapi(routes):
    "A minimal OpenAPI 3.1 document, built from the same `routes` the app serves rather than kept by hand."
    paths = {}
    for r in routes:
        if not isinstance(r, Route): continue
        for method in sorted(m for m in r.methods if m != 'HEAD'):
            paths.setdefault(r.path, {})[method.lower()] = {
                'operationId': r.name, 'responses': {'200': {'description': 'ok'}}}
    return {'openapi': '3.1.0', 'info': {'title': 'ramabana-serve', 'version': '1'}, 'paths': paths}

async def _openapi_route(request):
    return JSONResponse(request.app.state.openapi)


In [ ]:
#| export
def app(roots, model=None, cfg=None, **kw):
    "The Starlette app: sessions, prompts over SSE, plan, and approvals."
    routes = [
        Route('/sessions', _list_sessions, methods=['GET']),
        Route('/sessions', _create_session, methods=['POST']),
        Route('/sessions/{sid}', _close_session, methods=['DELETE']),
        Route('/sessions/{sid}/prompt', _prompt, methods=['POST']),
        Route('/sessions/{sid}/plan', _plan, methods=['GET']),
        Route('/sessions/{sid}/approval', _answer, methods=['POST']),
        Route('/openapi.json', _openapi_route, methods=['GET']),
    ]
    a = Starlette(routes=routes)
    a.state.sessions = Sessions(roots, model=model, cfg=Path(cfg) if cfg else None, **kw)
    a.state.openapi = _openapi(routes)
    return a


In [ ]:
#| export
@call_parse
def main(
    root: str = '.',                 # folders to serve, comma separated
    model: str = None,               # the default model new sessions route to
    host: str = '127.0.0.1',         # bind address; 127.0.0.1 keeps it off the network
    port: int = 4096,
    cfg: str = None,                 # config dir, for skills and extensions
):
    "Serve Ramabana sessions over HTTP and SSE."
    import uvicorn
    roots = [r.strip() for r in str(root).split(',') if r.strip()]
    uvicorn.run(app(roots, model=model, cfg=cfg), host=host, port=port)
